In [1]:
import os, glob
import sys
import settings
import json
import concurrent.futures
from google.cloud import pubsub_v1
import google.auth
import subprocess as sp
import time
import utils

In [7]:
list_res = utils.service.users().messages().list(userId='me', q='in:inbox', maxResults=1).execute()
messages = list_res.get('messages', [])
eid = messages[0]['id']
msg = utils.service.users().messages().get(
    userId='me', id=eid, format='full'
).execute()

In [18]:
msg

{'id': '19fda260c2538432',
 'threadId': '19fda260c2538432',
 'labelIds': ['UNREAD', 'IMPORTANT', 'CATEGORY_UPDATES', 'INBOX'],
 'snippet': 'Your Netflix temporary access code ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏',
 'payload': {'partId': '',
  'mimeType': 'multipart/alternative',
  'filename': '',
  'headers': [{'name': 'Delivered-To', 'value': 'ginoprasad3@gmail.com'},
   {'name': 'Received',
    'value': 'by 2002:a5d:4643:0:b0:47d:abb2:5f02 with SMTP id j3csp6167512wrs;        Thu, 6 Aug 2026 19:55:54 -0700 (PDT)'},
   {'name': 'X-Received',
    'value': 'by 2002:a05:6a20:cfa2:b0:3c3:875d:c52f with SMTP id adf61e73a8af0-3cb85dc28femr25796052637.10.1786071354563;        Thu, 06 Aug 2026 19:55:54 -0700 (PDT)'},
   {'name': 'ARC-Seal',
    'value': 'i=1; a=rsa-sha256; t=1786071354; cv=none;        d=google.com; s=arc-20260327;        b=KM3xsV0zj/LJgNEHpoSIW+9YT

In [14]:
'info@account.netflix.com' in str(msg)

True

In [10]:
msg['payload']

{'partId': '',
 'mimeType': 'multipart/alternative',
 'filename': '',
 'headers': [{'name': 'Delivered-To', 'value': 'ginoprasad3@gmail.com'},
  {'name': 'Received',
   'value': 'by 2002:a5d:4643:0:b0:47d:abb2:5f02 with SMTP id j3csp6167512wrs;        Thu, 6 Aug 2026 19:55:54 -0700 (PDT)'},
  {'name': 'X-Received',
   'value': 'by 2002:a05:6a20:cfa2:b0:3c3:875d:c52f with SMTP id adf61e73a8af0-3cb85dc28femr25796052637.10.1786071354563;        Thu, 06 Aug 2026 19:55:54 -0700 (PDT)'},
  {'name': 'ARC-Seal',
   'value': 'i=1; a=rsa-sha256; t=1786071354; cv=none;        d=google.com; s=arc-20260327;        b=KM3xsV0zj/LJgNEHpoSIW+9YTzvbY2GcnFYdl+Hygl6EUkooDMjfk03RWTQnllXwn9         zGb+fPXSHalb1rBPxXfHFWZY0KGT2PSbH8U9BaeZE2/iTPiWA8ogYKkfDJx6nwP5Qd9S         5O68FCIvozHdBADk6QAVsdjUXyiC8UFQ8tP5LwAZv63YYbfcW/ytpc6anIUnl4zajmFE         dbtIzfDQfhl+gt2wHIBnWVxwrO4ImAgigXkDmSvPWUwVGiv/eDvcFJca17z9f22buPHs         +bByQFMH5t9O/59KRUmMYIp3SY0+O0j5IHQST3zEICidrMd1FnSzQPMaC9387aHBJ6k/         EYgQ==

In [ ]:
credentials, default_project = google.auth.default(
    quota_project_id=settings.project_id
)

subscriber = pubsub_v1.SubscriberClient(credentials=credentials)
subscription_path = subscriber.subscription_path(settings.project_id, settings.subscription_id)

def callback(message):
    message.ack()
    data = json.loads(message.data.decode('utf-8'))
    list_res = utils.service.users().messages().list(userId='me', q='in:inbox', maxResults=1).execute()
    messages = list_res.get('messages', [])
    eid = messages[0]['id']
    
    if eid == utils.get_last_email_id():
        return
    utils.update_last_email_id(eid)

    msg = utils.service.users().messages().get(
        userId='me', id=eid, format='minimal'
    ).execute()

    labels = msg.get('labelIds', [])
    if 'INBOX' in labels and 'SENT' not in labels:
        cmd = f"{settings.ttab_path} '{os.getcwd()}/forward_email.py {eid}; exit'"
        sp.run(cmd, shell=True)


streaming_pull_future = subscriber.subscribe(subscription_path, callback=callback)

try:
    streaming_pull_future.result(timeout=settings.TIMEOUT)
except concurrent.futures.TimeoutError:
    streaming_pull_future.cancel()
    print(f"{settings.TIMEOUT} second window finished. Exiting gracefully.")




